# Reversible Circuits Project: Phase 4

Reversible 4-bit adders, and the rule-based optimizer applied to them.

- **Rules and optimizer:** Feng and Li, *A Complete Set of Transformation Rules for Reversible Circuits*, IEEE TCAD, 2026.
- **Adder design:** Kumar et al., *An Optimised Design of a 4-bit Carry Look-Ahead Adder Using Novel Reversible Logic Gates*, IACIS 2024.

**How to run:** choose **Runtime > Run all** from the menu. The whole notebook takes about a minute.

| Step | What it does |
|---|---|
| 1 | Creates the earlier files `revcircuit.py`, `rules.py`, `optimizer.py` (unchanged) |
| 2 | Creates `adders.py`: Feynman, Peres, TSG and R2 blocks, the carry look-ahead adder, a ripple-carry adder |
| 3 | Creates `bitsim.py`: fast simulation of all inputs at once |
| 4 | Creates `verify_phase4.py`, the Phase 4 tests and experiments |
| 5 | Runs them |
| 6 | An example you can edit and re-run |

**Read this first.** The carry look-ahead netlist is a *reconstruction*. The adder paper gives a block diagram, not a netlist, and its diagram lets one signal feed several blocks, which a reversible circuit cannot do without copy gates. The circuit here follows the paper's blocks and equations and adds one Feynman copy gate wherever a signal is needed twice. It therefore has 21 blocks and 15 ancilla inputs, against the 20 and 14 the paper reports.

## Step 1: Earlier files (unchanged)

In [ ]:
%%writefile revcircuit.py
"""Reversible circuit model and simulator (Phase 1).

Gates are mixed-polarity multiple-control Toffoli (MPMCT) gates, written
G[P, N, q] as in Feng and Li, "A Complete Set of Transformation Rules for
Reversible Circuits", IEEE TCAD, 2026:

    P  set of positive control lines
    N  set of negative control lines
    q  target line

The gate flips q when every line in P is 1 and every line in N is 0.
Special cases: X[q] = G[{}, {}, q], CNOT[p, q] = G[{p}, {}, q], and a
Toffoli gate is G[{a, b}, {}, q].

A basis state is stored as an integer whose bit i is the value of line q_i.
A circuit is a list of gates; the leftmost gate executes first.
"""
from __future__ import annotations

import random
from dataclasses import dataclass
from itertools import product
from typing import Iterable, Iterator, List, Optional, Sequence, Tuple


# ---------------------------------------------------------------------------
# Gate
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class Gate:
    """One MPMCT gate G[pos, neg, target]."""

    pos: frozenset
    neg: frozenset
    target: int

    def __post_init__(self) -> None:
        object.__setattr__(self, "pos", frozenset(self.pos))
        object.__setattr__(self, "neg", frozenset(self.neg))
        if self.pos & self.neg:
            raise ValueError("a line cannot be both a positive and a negative control")
        if self.target in self.pos or self.target in self.neg:
            raise ValueError("the target line cannot also be a control")
        if min(self.lines) < 0:
            raise ValueError("line numbers must be non-negative")
        # Bit masks used by the simulator.
        object.__setattr__(self, "_pmask", sum(1 << l for l in self.pos))
        object.__setattr__(self, "_nmask", sum(1 << l for l in self.neg))
        object.__setattr__(self, "_tmask", 1 << self.target)

    @property
    def controls(self) -> frozenset:
        return self.pos | self.neg

    @property
    def lines(self) -> frozenset:
        return self.pos | self.neg | {self.target}

    def apply(self, state: int) -> int:
        """Return the state after this gate acts on `state`."""
        if (state & self._pmask) == self._pmask and not (state & self._nmask):
            return state ^ self._tmask
        return state

    def __str__(self) -> str:
        t = self.target
        if not self.neg:
            if not self.pos:
                return f"X[q{t}]"
            if len(self.pos) == 1:
                (c,) = self.pos
                return f"CNOT[q{c},q{t}]"

        def fmt(s: frozenset) -> str:
            return "{" + ",".join(f"q{l}" for l in sorted(s)) + "}"

        return f"G[{fmt(self.pos)},{fmt(self.neg)},q{t}]"


def G(pos: Iterable[int], neg: Iterable[int], target: int) -> Gate:
    """General gate G[P, N, q]."""
    return Gate(frozenset(pos), frozenset(neg), target)


def X(q: int) -> Gate:
    """NOT gate on line q."""
    return Gate(frozenset(), frozenset(), q)


def CNOT(control: int, target: int) -> Gate:
    """Controlled-NOT (Feynman gate)."""
    return Gate(frozenset({control}), frozenset(), target)


def TOF(c1: int, c2: int, target: int) -> Gate:
    """Toffoli gate with two positive controls."""
    return Gate(frozenset({c1, c2}), frozenset(), target)


def MCT(controls: Iterable[int], target: int) -> Gate:
    """Multiple-control Toffoli gate with positive controls only."""
    return Gate(frozenset(controls), frozenset(), target)


# ---------------------------------------------------------------------------
# Circuit
# ---------------------------------------------------------------------------
class Circuit:
    """A reversible circuit on n lines: an ordered list of gates."""

    def __init__(self, n: int, gates: Iterable[Gate] = ()) -> None:
        self.n = n
        self.gates: List[Gate] = list(gates)
        for g in self.gates:
            if max(g.lines) >= n:
                raise ValueError(f"gate {g} uses a line outside q0..q{n - 1}")

    # -- container behaviour ------------------------------------------------
    def __len__(self) -> int:
        return len(self.gates)

    def __getitem__(self, i: int) -> Gate:
        return self.gates[i]

    def __iter__(self) -> Iterator[Gate]:
        return iter(self.gates)

    def __eq__(self, other: object) -> bool:
        """Structural equality: same lines and the same gate list."""
        return isinstance(other, Circuit) and self.n == other.n and self.gates == other.gates

    def __add__(self, other: "Circuit") -> "Circuit":
        """Concatenation: self executes first, then other."""
        if self.n != other.n:
            raise ValueError("circuits must have the same number of lines")
        return Circuit(self.n, self.gates + other.gates)

    def __str__(self) -> str:
        return " ".join(str(g) for g in self.gates) if self.gates else "(empty)"

    def copy(self) -> "Circuit":
        return Circuit(self.n, self.gates)

    def replace(self, i: int, j: int, new_gates: Sequence[Gate]) -> "Circuit":
        """Return a new circuit with gates[i:j] replaced by new_gates."""
        return Circuit(self.n, self.gates[:i] + list(new_gates) + self.gates[j:])

    def inverse(self) -> "Circuit":
        """Every gate is its own inverse, so the inverse is the reversed list."""
        return Circuit(self.n, reversed(self.gates))

    # -- simulation ---------------------------------------------------------
    def apply(self, state: int) -> int:
        for g in self.gates:
            state = g.apply(state)
        return state

    def permutation(self) -> Tuple[int, ...]:
        """The reversible function computed, as a tuple over all 2^n states."""
        return tuple(self.apply(s) for s in range(1 << self.n))

    def equivalent(self, other: "Circuit") -> bool:
        """Functional equivalence: both circuits compute the same function."""
        return self.n == other.n and self.permutation() == other.permutation()

    # -- display ------------------------------------------------------------
    def _bits(self, state: int) -> str:
        return " ".join(str((state >> i) & 1) for i in range(self.n))

    def truth_table(self) -> str:
        head = " ".join(f"q{i}" for i in range(self.n))
        rows = [f"{head} | {head}"]
        for s, t in enumerate(self.permutation()):
            pad = "  ".join(self._bits(s).split())
            out = "  ".join(self._bits(t).split())
            rows.append(f" {pad} |  {out}")
        return "\n".join(rows)

    def draw(self) -> str:
        """Text drawing: circled plus = target, filled dot = positive control, open dot = negative control."""
        width = len(f"q{self.n - 1}")
        rows = []
        for r in range(self.n):
            cells = []
            for g in self.gates:
                lo, hi = min(g.lines), max(g.lines)
                if r == g.target:
                    ch = "\u2295"  # target
                elif r in g.pos:
                    ch = "\u25cf"  # filled dot
                elif r in g.neg:
                    ch = "\u25cb"  # open dot
                elif lo < r < hi:
                    ch = "\u253c"  # wire crossing a gate
                else:
                    ch = "\u2500"
                cells.append("\u2500" + ch + "\u2500")
            rows.append(f"q{r}:".ljust(width + 2) + "\u2500" + "".join(cells) + "\u2500")
        return "\n".join(rows)


# ---------------------------------------------------------------------------
# Helpers for experiments
# ---------------------------------------------------------------------------
def pn_choices(lines: Sequence[int]) -> Iterator[Tuple[frozenset, frozenset]]:
    """All ways to make each given line absent, a positive or a negative control."""
    for states in product((0, 1, 2), repeat=len(lines)):
        pos = frozenset(l for l, s in zip(lines, states) if s == 1)
        neg = frozenset(l for l, s in zip(lines, states) if s == 2)
        yield pos, neg


def all_gates(n: int) -> List[Gate]:
    """Every MPMCT gate on n lines: n * 3^(n-1) gates."""
    gates = []
    for target in range(n):
        others = [l for l in range(n) if l != target]
        for pos, neg in pn_choices(others):
            gates.append(Gate(pos, neg, target))
    return gates


def random_gate(n: int, rng: Optional[random.Random] = None) -> Gate:
    rng = rng or random
    target = rng.randrange(n)
    pos, neg = set(), set()
    for l in range(n):
        if l == target:
            continue
        s = rng.randrange(3)
        if s == 1:
            pos.add(l)
        elif s == 2:
            neg.add(l)
    return Gate(frozenset(pos), frozenset(neg), target)


def random_circuit(n: int, length: int, rng: Optional[random.Random] = None) -> Circuit:
    return Circuit(n, [random_gate(n, rng) for _ in range(length)])

In [ ]:
%%writefile rules.py
"""Transformation rules for reversible circuits (Phase 1).

Rules 1 to 5 are the complete rule set RC of Feng and Li (IEEE TCAD, 2026).
Rules 6 to 10 are the rules the paper derives from RC.

Every rewrite function takes a circuit and a position i and returns a NEW
circuit with the rule applied at position i, or None when the rule does not
match there. The input circuit is never modified.

For rules stated on gates rather than positions, a `ruleK_gates` or
`ruleK_sides` helper returns the gate lists on each side of the rule.
"""
from __future__ import annotations

from itertools import combinations
from typing import Iterable, List, Optional, Sequence, Tuple

from revcircuit import CNOT, Circuit, G, Gate, X


# ===========================================================================
# Rule 1:  A A  ==  (empty)
# ===========================================================================
def rule1_cancel(c: Circuit, i: int) -> Optional[Circuit]:
    """Delete two adjacent identical gates at positions i and i+1."""
    if 0 <= i and i + 1 < len(c) and c[i] == c[i + 1]:
        return c.replace(i, i + 2, [])
    return None


def rule1_insert(c: Circuit, i: int, gate: Gate) -> Circuit:
    """Rule 1 read right to left: insert the pair `gate gate` before position i."""
    return c.replace(i, i, [gate, gate])


# ===========================================================================
# Rule 2:  G[P, N+{p}, q] G[P+{p}, N, q]  ==  G[P, N, q]
# ===========================================================================
def rule2_gates(gate: Gate, p: int) -> List[Gate]:
    """The pair A0, A1 that Rule 2 merges into `gate`, using extra line p."""
    if p in gate.lines:
        raise ValueError("p must be a line the gate does not use")
    return [G(gate.pos, gate.neg | {p}, gate.target), G(gate.pos | {p}, gate.neg, gate.target)]


def rule2_merge(c: Circuit, i: int) -> Optional[Circuit]:
    """Merge gates i and i+1 when they are A0 A1 in the order of the paper."""
    if i < 0 or i + 1 >= len(c):
        return None
    a0, a1 = c[i], c[i + 1]
    extra = a1.pos - a0.pos
    if a0.target != a1.target or len(extra) != 1:
        return None
    (p,) = extra
    merged = G(a1.pos - {p}, a1.neg, a1.target)
    if rule2_gates(merged, p) == [a0, a1]:
        return c.replace(i, i + 2, [merged])
    return None


def rule2_split(c: Circuit, i: int, p: int) -> Optional[Circuit]:
    """Rule 2 read right to left: split gate i into A0 A1 on unused line p."""
    if not (0 <= i < len(c)) or not (0 <= p < c.n) or p in c[i].lines:
        return None
    return c.replace(i, i + 1, rule2_gates(c[i], p))


# ===========================================================================
# Rule 3:  A B == B A  when a control has opposite polarity in A and B
# ===========================================================================
def rule3_condition(a: Gate, b: Gate) -> bool:
    return bool(a.pos & b.neg) or bool(b.pos & a.neg)


def rule3_commute(c: Circuit, i: int) -> Optional[Circuit]:
    """Swap gates i and i+1 when they share a control of opposite polarity."""
    if 0 <= i and i + 1 < len(c) and rule3_condition(c[i], c[i + 1]):
        return c.replace(i, i + 2, [c[i + 1], c[i]])
    return None


# ===========================================================================
# Rule 4:  A B A C1 == C2 A B A,  A = CNOT[p,q], B = CNOT[q,p]
#          (moving a swap of p and q through a gate)
# ===========================================================================
def rule4_sides(p: int, q: int, P: Iterable[int], N: Iterable[int],
                negative: bool = False) -> Tuple[List[Gate], List[Gate]]:
    """Both sides of Rule 4.

    negative=False is condition (a): C1 has q as positive control, target p.
    negative=True  is condition (b): C1 has q as negative control, target p.
    """
    P, N = frozenset(P), frozenset(N)
    a, b = CNOT(p, q), CNOT(q, p)
    if negative:
        c1, c2 = G(P, N | {q}, p), G(P, N | {p}, q)
    else:
        c1, c2 = G(P | {q}, N, p), G(P | {p}, N, q)
    return [a, b, a, c1], [c2, a, b, a]


def _is_cnot(g: Gate) -> bool:
    return len(g.pos) == 1 and not g.neg


def rule4_apply(c: Circuit, i: int) -> Optional[Circuit]:
    """Apply Rule 4 to the four gates starting at i, in either direction."""
    if i < 0 or i + 4 > len(c):
        return None
    seg = c.gates[i:i + 4]
    # Left to right:  A B A C1  ->  C2 A B A
    if _is_cnot(seg[0]):
        (p,) = seg[0].pos
        q = seg[0].target
        c1 = seg[3]
        if c1.target == p and q in c1.controls:
            lhs, rhs = rule4_sides(p, q, c1.pos - {q}, c1.neg - {q}, q in c1.neg)
            if seg == lhs:
                return c.replace(i, i + 4, rhs)
    # Right to left:  C2 A B A  ->  A B A C1
    if _is_cnot(seg[1]):
        (p,) = seg[1].pos
        q = seg[1].target
        c2 = seg[0]
        if c2.target == q and p in c2.controls:
            lhs, rhs = rule4_sides(p, q, c2.pos - {p}, c2.neg - {p}, p in c2.neg)
            if seg == rhs:
                return c.replace(i, i + 4, lhs)
    return None


# ===========================================================================
# Rule 5:  A0 A1 B1..Bm..B1 A1 A0  ==  B'1..B'm..B'1
#          (changes the polarity of control q0)
# ===========================================================================
def rule5_sides(P: Iterable[int], N: Iterable[int], Q: Sequence[int],
                q0: int) -> Tuple[List[Gate], List[Gate]]:
    """Both sides of Rule 5 for control sets P, N, ordered list Q and line q0."""
    P, N, Q = frozenset(P), frozenset(N), list(Q)
    Qs = frozenset(Q)
    if not Q or len(Qs) != len(Q):
        raise ValueError("Q must be a non-empty list of distinct lines")
    a0 = G(P, N | Qs, q0)
    a1 = G(P | Qs, N, q0)
    b = [G(P | {q0} | frozenset(Q[k + 1:]), N | frozenset(Q[:k]), Q[k]) for k in range(len(Q))]
    bp = [G(P | frozenset(Q[k + 1:]), N | {q0} | frozenset(Q[:k]), Q[k]) for k in range(len(Q))]
    lhs = [a0, a1] + b + b[-2::-1] + [a1, a0]
    rhs = bp + bp[-2::-1]
    return lhs, rhs


def rule5_apply(c: Circuit, i: int) -> Optional[Circuit]:
    """Left to right: replace the 2m+3 gate pattern starting at i by 2m-1 gates."""
    if i < 0 or i + 1 >= len(c):
        return None
    a0, a1 = c[i], c[i + 1]
    Qs = a1.pos - a0.pos
    if a0.target != a1.target or not Qs or a0.neg - a1.neg != Qs:
        return None
    if a0.pos - a1.pos or a1.neg - a0.neg:
        return None
    m = len(Qs)
    end = i + 2 * m + 3
    if end > len(c):
        return None
    order = [g.target for g in c.gates[i + 2:i + 2 + m]]
    if frozenset(order) != Qs:
        return None
    lhs, rhs = rule5_sides(a0.pos, a1.neg, order, a0.target)
    if c.gates[i:end] == lhs:
        return c.replace(i, end, rhs)
    return None


def rule5_reverse(c: Circuit, i: int, Q: Sequence[int], q0: int) -> Optional[Circuit]:
    """Right to left: expand B'1..B'm..B'1 at i, given the list Q and line q0."""
    Q = list(Q)
    end = i + 2 * len(Q) - 1
    if i < 0 or not Q or end > len(c):
        return None
    b1 = c[i]
    if b1.target != Q[0] or q0 not in b1.neg:
        return None
    try:
        lhs, rhs = rule5_sides(b1.pos - frozenset(Q[1:]), b1.neg - {q0}, Q, q0)
    except ValueError:
        return None
    if c.gates[i:end] == rhs:
        return c.replace(i, end, lhs)
    return None


# ===========================================================================
# Derived rules 6 to 10
# ===========================================================================
# Rule 6:  G[P,N,q] == product over all subsets S of Q of G[P+S, N+(Q-S), q]
def rule6_gates(gate: Gate, Q: Iterable[int]) -> List[Gate]:
    Q = sorted(set(Q))
    if set(Q) & gate.lines:
        raise ValueError("Q must not overlap the lines of the gate")
    out = []
    for r in range(len(Q) + 1):
        for sub in combinations(Q, r):
            out.append(G(gate.pos | set(sub), gate.neg | (set(Q) - set(sub)), gate.target))
    return out


def rule6_expand(c: Circuit, i: int, Q: Iterable[int]) -> Optional[Circuit]:
    """Expand gate i over all polarity patterns of the unused lines Q."""
    Q = set(Q)
    if not (0 <= i < len(c)) or not Q or Q & c[i].lines or max(Q) >= c.n or min(Q) < 0:
        return None
    return c.replace(i, i + 1, rule6_gates(c[i], Q))


# Rule 7:  two gates with the same target commute
def rule7_commute(c: Circuit, i: int) -> Optional[Circuit]:
    if 0 <= i and i + 1 < len(c) and c[i].target == c[i + 1].target:
        return c.replace(i, i + 2, [c[i + 1], c[i]])
    return None


# Rule 8:  G[P,N,q] == X[N] G[P+N, {}, q] X[N]
def rule8_gates(gate: Gate) -> List[Gate]:
    xs = [X(l) for l in sorted(gate.neg)]
    return xs + [G(gate.pos | gate.neg, (), gate.target)] + xs


def rule8_expand(c: Circuit, i: int) -> Optional[Circuit]:
    """Replace the negative controls of gate i by X gates before and after."""
    if 0 <= i < len(c) and c[i].neg:
        return c.replace(i, i + 1, rule8_gates(c[i]))
    return None


def rule8_collapse(c: Circuit, i: int) -> Optional[Circuit]:
    """Right to left: absorb X..X B X..X starting at i into one gate."""
    k = 0
    while i + k < len(c) and not c[i + k].controls:
        k += 1
    if i < 0 or k == 0 or i + 2 * k + 1 > len(c):
        return None
    b = c[i + k]
    lines = frozenset(g.target for g in c.gates[i:i + k])
    if len(lines) != k or b.neg or not lines <= b.pos:
        return None
    merged = G(b.pos - lines, lines, b.target)
    if c.gates[i:i + 2 * k + 1] == rule8_gates(merged):
        return c.replace(i, i + 2 * k + 1, [merged])
    return None


# Rule 9:  A B1 == B2 A, where A targets p and B1, B2 differ in the polarity of p
def _rule9_pair(a: Gate, b: Gate) -> Optional[Gate]:
    """If A can pass B by Rule 9, return B with the polarity of A's target flipped."""
    p = a.target
    if p not in b.controls:
        return None
    if not (a.pos <= b.pos - {p} and a.neg <= b.neg - {p}):
        return None
    if p in b.pos:
        return G(b.pos - {p}, b.neg | {p}, b.target)
    return G(b.pos | {p}, b.neg - {p}, b.target)


def rule9_apply(c: Circuit, i: int) -> Optional[Circuit]:
    """Move a gate past its neighbour, flipping one control polarity in the neighbour."""
    if i < 0 or i + 1 >= len(c):
        return None
    g1, g2 = c[i], c[i + 1]
    flipped = _rule9_pair(g1, g2)          # A B1 -> B2 A
    if flipped is not None:
        return c.replace(i, i + 2, [flipped, g1])
    flipped = _rule9_pair(g2, g1)          # B2 A -> A B1
    if flipped is not None:
        return c.replace(i, i + 2, [g2, flipped])
    return None


# Rule 10:  A B A == B A B,  A = G[P+{p}, {}, q],  B = G[P+{q}, {}, p]
def rule10_sides(P: Iterable[int], p: int, q: int) -> Tuple[List[Gate], List[Gate]]:
    P = frozenset(P)
    a, b = G(P | {p}, (), q), G(P | {q}, (), p)
    return [a, b, a], [b, a, b]


def rule10_apply(c: Circuit, i: int) -> Optional[Circuit]:
    if i < 0 or i + 3 > len(c):
        return None
    a, b = c[i], c[i + 1]
    p, q = b.target, a.target
    if a.neg or b.neg or p == q or p not in a.pos or q not in b.pos:
        return None
    if a.pos - {p} != b.pos - {q}:
        return None
    lhs, rhs = rule10_sides(a.pos - {p}, p, q)
    if c.gates[i:i + 3] == lhs:
        return c.replace(i, i + 3, rhs)
    return None

In [ ]:
%%writefile optimizer.py
"""Heuristic rule-based optimizer for reversible circuits (Phase 3).

The optimizer rewrites a circuit with the transformation rules of Feng and
Li (IEEE TCAD, 2026) until the cost stops falling. One search step takes a
gate, moves it along the circuit, and looks for a partner it can cancel or
merge with.

Moves that remove gates
    Rule 1          A A                      ->  (nothing)
    Rule 2          G[P,N+p,q] G[P+p,N,q]    ->  G[P,N,q]
    Rule 2 + 1      G[P,N,q]   G[P+p,N,q]    ->  G[P,N+p,q]      ("absorb")
    Rule 5          the 2m+3 gate pattern    ->  2m-1 gates

Moves that carry a gate past its neighbour
    Rule 7          same target: the gates commute
    Rule 3          a shared control of opposite polarity: they commute
    D1 (derived)    neither target is a control of the other: they commute
    Rule 9          the gate passes; a control polarity flips in one gate
    D2 (derived)    the gate passes and leaves one extra gate behind

D1 and D2 are not listed in the paper. Both are sound (verify_phase3.py
checks every instance on 2 to 4 lines), so by the completeness theorem of
the paper they follow from Rules 1 to 5. Rule 9 is the special case of D2
in which the extra gate merges with its neighbour.

Every accepted rewrite strictly lowers the objective, so the search stops.
With check=True each rewrite is also verified by simulating the replaced
gates and the new gates on the lines they use.
"""
from __future__ import annotations

import random
from collections import Counter
from dataclasses import dataclass, field
from typing import Callable, List, Optional, Sequence, Tuple

from revcircuit import Circuit, G, Gate
import rules as R

CostFn = Callable[[Gate], int]


# ---------------------------------------------------------------------------
# Cost model
# ---------------------------------------------------------------------------
def quantum_cost(gate: Gate) -> int:
    """Quantum cost of one mixed-polarity multiple-control Toffoli gate.

    Controls:  0   1   2   3    4    5   ...   c
    Cost:      1   1   5   13   29   61  ...   2^(c+1) - 3

    This is the count of elementary gates in the decomposition of Barenco et
    al. (1995) without ancilla lines. Negative controls are free, except that
    a gate whose controls are ALL negative costs one more. The model is a
    stated assumption of this project; pass another function to change it.
    """
    c = len(gate.pos) + len(gate.neg)
    cost = 1 if c <= 1 else (1 << (c + 1)) - 3
    if c >= 1 and not gate.pos:
        cost += 1
    return cost


def circuit_cost(circuit, cost: CostFn = quantum_cost) -> int:
    """Total quantum cost of a circuit or a list of gates."""
    return sum(cost(g) for g in circuit)


# ---------------------------------------------------------------------------
# Elementary moves
# ---------------------------------------------------------------------------
def reduce_pair(a: Gate, b: Gate) -> Optional[Tuple[List[Gate], str]]:
    """Replace two gates with the same target by at most one gate, if possible."""
    if a.target != b.target:
        return None
    if a == b:
        return [], "Rule 1"
    la, lb = a.controls, b.controls
    if la == lb:
        differ = a.pos ^ b.pos                      # lines with opposite polarity
        if len(differ) == 1:
            (p,) = differ
            return [G(a.pos - {p}, a.neg - {p}, a.target)], "Rule 2"
    elif len(la ^ lb) == 1:
        big, small = (a, b) if len(la) > len(lb) else (b, a)
        (p,) = big.controls - small.controls
        if big.pos - {p} == small.pos and big.neg - {p} == small.neg:
            return [_flip(big, p)], "Rule 2 + 1"
    return None


def _flip(gate: Gate, p: int) -> Gate:
    """The same gate with the polarity of control p reversed."""
    if p in gate.pos:
        return G(gate.pos - {p}, gate.neg | {p}, gate.target)
    return G(gate.pos | {p}, gate.neg - {p}, gate.target)


def commute_tag(a: Gate, b: Gate) -> Optional[str]:
    """Name of the rule by which two adjacent gates commute, or None."""
    if a.target == b.target:
        return "Rule 7"
    if (a.pos & b.neg) or (a.neg & b.pos):
        return "Rule 3"
    if a.target not in b.controls and b.target not in a.controls:
        return "D1"
    return None


def induced_gate(a: Gate, b: Gate) -> Optional[Gate]:
    """The extra gate of rule D2.

    A = G[P1,N1,p] writes line p, and B = G[..p.., q] reads it. Then
        A B  ==  B B'' A      and      B A  ==  A B'' B
    with B'' = G[P1 + P2, N1 + N2, q], where P2, N2 are the controls of B
    other than p. Returns None when the rule does not apply.
    """
    p, q = a.target, b.target
    if p not in b.controls or q in a.controls:
        return None
    pos, neg = a.pos | (b.pos - {p}), a.neg | (b.neg - {p})
    if pos & neg:
        return None                                  # then Rule 3 applies instead
    return G(pos, neg, q)


def pass_gate(moving: Gate, fixed: Gate) -> Optional[Tuple[List[Gate], Gate, str, int]]:
    """Carry `moving` from the left of `fixed` to its right.

    Returns (gates left behind, the moving gate afterwards, rule, extra gates)
    or None when neither Rule 9 nor D2 applies.
    """
    extra = induced_gate(moving, fixed)              # moving writes, fixed reads
    if extra is not None:
        merged = reduce_pair(fixed, extra)
        if merged is not None:
            return merged[0], moving, "Rule 9", 0
        return [fixed, extra], moving, "D2", 1
    extra = induced_gate(fixed, moving)              # fixed writes, moving reads
    if extra is not None:
        merged = reduce_pair(extra, moving)
        if merged is not None:
            return [fixed], merged[0][0], "Rule 9", 0
        return [fixed, extra], moving, "D2", 1
    return None


# ---------------------------------------------------------------------------
# Checking one rewrite
# ---------------------------------------------------------------------------
def same_function(old: Sequence[Gate], new: Sequence[Gate],
                  rng: Optional[random.Random] = None, limit: int = 12) -> bool:
    """True when two gate lists act identically on the lines they use.

    Exhaustive for up to `limit` lines; beyond that, 4096 random states.
    """
    lines = sorted(set().union(*(g.lines for g in list(old) + list(new)))) if (old or new) else []
    index = {l: i for i, l in enumerate(lines)}

    def remap(gates):
        return [G([index[l] for l in g.pos], [index[l] for l in g.neg], index[g.target])
                for g in gates]

    a, b = remap(old), remap(new)
    k = len(lines)
    if k <= limit:
        states = range(1 << k)
    else:
        rng = rng or random.Random(0)
        states = [rng.getrandbits(k) for _ in range(4096)]
    for s in states:
        x = y = s
        for g in a:
            x = g.apply(x)
        for g in b:
            y = g.apply(y)
        if x != y:
            return False
    return True


# ---------------------------------------------------------------------------
# The optimizer
# ---------------------------------------------------------------------------
@dataclass
class OptimizationResult:
    original: Circuit
    circuit: Circuit
    rule_counts: Counter = field(default_factory=Counter)
    rewrites: int = 0
    passes: int = 0
    cost_fn: CostFn = quantum_cost

    @property
    def gates_before(self) -> int:
        return len(self.original)

    @property
    def gates_after(self) -> int:
        return len(self.circuit)

    @property
    def cost_before(self) -> int:
        return circuit_cost(self.original, self.cost_fn)

    @property
    def cost_after(self) -> int:
        return circuit_cost(self.circuit, self.cost_fn)

    def summary(self) -> str:
        def change(before, after):
            pct = 100.0 * (before - after) / before if before else 0.0
            return f"{before} -> {after} ({pct:.1f}% lower)"

        lines = [f"Gate count:   {change(self.gates_before, self.gates_after)}",
                 f"Quantum cost: {change(self.cost_before, self.cost_after)}",
                 f"Rewrites accepted: {self.rewrites} in {self.passes} passes"]
        if self.rule_counts:
            lines.append("Rule usage: " + ", ".join(
                f"{name} x{count}" for name, count in sorted(self.rule_counts.items())))
        return "\n".join(lines)


class _Search:
    def __init__(self, objective: str, window: int, extra_gates: int,
                 cost: CostFn, check: bool) -> None:
        if objective not in ("cost", "gates"):
            raise ValueError('objective must be "cost" or "gates"')
        self.objective, self.window, self.extra_gates = objective, window, extra_gates
        self.cost, self.check = cost, check
        self.counts: Counter = Counter()
        self.rewrites = 0
        self.rng = random.Random(0)

    def score(self, gates: Sequence[Gate]) -> Tuple[int, int]:
        qc, count = sum(self.cost(g) for g in gates), len(gates)
        return (qc, count) if self.objective == "cost" else (count, qc)

    def walk(self, gates: List[Gate], i: int):
        """Move gate i to the right; return the best improving rewrite found."""
        moving, left, tags, growth = gates[i], [], [], 0
        best = None
        for k in range(i + 1, min(len(gates), i + 1 + self.window)):
            fixed = gates[k]
            reduced = reduce_pair(moving, fixed)
            if reduced is not None:
                new = left + reduced[0]
                gain = tuple(o - n for o, n in zip(self.score(gates[i:k + 1]), self.score(new)))
                if gain > (0, 0) and (best is None or gain > best[0]):
                    best = (gain, k, new, tags + [reduced[1]])
            tag = commute_tag(moving, fixed)
            if tag is not None:
                left, tags = left + [fixed], tags + [tag]
                continue
            passed = pass_gate(moving, fixed)
            if passed is None or growth + passed[3] > self.extra_gates:
                break
            left, moving = left + passed[0], passed[1]
            tags, growth = tags + [passed[2]], growth + passed[3]
        return best

    def sweep(self, gates: List[Gate]) -> bool:
        """One left-to-right sweep. Returns True when the circuit changed."""
        changed, i = False, 0
        while i < len(gates):
            best = self.walk(gates, i)
            if best is None:
                i += 1
                continue
            _, k, new, tags = best
            if self.check and not same_function(gates[i:k + 1], new, self.rng):
                raise AssertionError("optimizer produced a non-equivalent rewrite")
            gates[i:k + 1] = new
            self.counts.update(tags)
            self.rewrites += 1
            changed = True
        return changed

    def rule5(self, n: int, gates: List[Gate]) -> bool:
        """Apply Rule 5 wherever its pattern appears literally."""
        changed, i = False, 0
        while i < len(gates):
            new = R.rule5_apply(Circuit(n, gates), i)
            if new is not None and self.score(new.gates) < self.score(gates):
                gates[:] = new.gates
                self.counts["Rule 5"] += 1
                self.rewrites += 1
                changed = True
            else:
                i += 1
        return changed


def optimize(circuit: Circuit, objective: str = "cost", window: int = 50,
             extra_gates: int = 1, cost: CostFn = quantum_cost,
             check: bool = True, max_passes: int = 100) -> OptimizationResult:
    """Reduce a reversible circuit with the transformation rules.

    objective    "cost" minimises quantum cost, then gate count;
                 "gates" minimises gate count, then quantum cost.
    window       how many gates a moving gate may travel past.
    extra_gates  how many extra gates (rule D2) one move may leave behind.
    check        verify every rewrite by simulation on the lines it touches.
    """
    search = _Search(objective, window, extra_gates, cost, check)
    gates = list(circuit.gates)
    passes = 0
    while passes < max_passes:
        passes += 1
        changed = search.sweep(gates)
        # The same search on the reversed circuit moves gates to the left.
        # Each gate is its own inverse, so the reversed circuit is the
        # inverse circuit and every rule stays valid.
        gates.reverse()
        changed |= search.sweep(gates)
        gates.reverse()
        changed |= search.rule5(circuit.n, gates)
        if not changed:
            break
    return OptimizationResult(circuit, Circuit(circuit.n, gates), search.counts,
                              search.rewrites, passes, cost)

## Step 2: The adders

| Block | Function | As Toffoli-type gates |
|---|---|---|
| Feynman | (A, B) → (A, A⊕B) | 1 CNOT |
| Peres | (A, B, C) → (A, A⊕B, AB⊕C) | 1 Toffoli + 1 CNOT |
| TSG | (A, B, C, D) → (A, Q, Q⊕D, QD⊕AB⊕C), Q = A'C'⊕B' | 3 Toffoli-type + 2 CNOT |
| R2 | (A, B, C, D) → (A, B, C, D⊕A⊕B⊕C) | 3 CNOT |

The carry look-ahead adder computes

- P_i = A_i ⊕ B_i and G_i = A_i·B_i (Peres), S_i = P_i ⊕ C_i (Feynman)
- C1 = G0 ⊕ P0·C0
- C2 = G1 ⊕ P1·G0 ⊕ P1·P0·C0 (three-input XOR by the TSG gate)
- C3 = G2 ⊕ P2·G1 ⊕ P2·P1·G0 ⊕ P2·P1·P0·C0 (four-input XOR by the R2 gate)

The ripple-carry adder uses two Peres gates per bit.

In [ ]:
%%writefile adders.py
"""Reversible adders built from Feynman, Peres, TSG and R2 gates (Phase 4).

Two 4-bit adders are provided, each as a circuit of Toffoli-type gates:

  cla_adder()   carry look-ahead adder in the style of Kumar et al.,
                "An Optimised Design of a 4-bit Carry Look-Ahead Adder Using
                Novel Reversible Logic Gates", IACIS 2024.
  rca_adder()   ripple-carry adder from two Peres gates per bit.

IMPORTANT: the CLA netlist is a RECONSTRUCTION. The paper gives a block
diagram (its Fig. 6), not a netlist, and the diagram lets one signal feed
several blocks, which a reversible circuit cannot do without copy gates.
The circuit here follows the paper's building blocks and equations:

    PGA block   Peres(A_i, B_i, 0) gives P_i = A_i xor B_i and G_i = A_i B_i;
                Feynman(P_i, C_i) gives S_i = P_i xor C_i
    C1 = G0 xor P0 C0                               Peres + Feynman
    C2 = G1 xor P1 G0 xor P1 P0 C0                  Peres gates + TSG
    C3 = G2 xor P2 G1 xor P2 P1 G0 xor P2 P1 P0 C0  Peres gates + R2

and adds a Feynman copy gate wherever a signal is needed twice.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List, Sequence, Tuple

from revcircuit import CNOT, Circuit, G, Gate, TOF

# Quantum cost of each block as stated in the adder paper.
PAPER_BLOCK_COST = {"Feynman": 1, "Peres": 4, "TSG": 17, "R2": 5}

# Figures the adder paper reports for its own design (its Table I).
PAPER_REPORTED = {"gates": 20, "ancilla": 14, "garbage": 14, "cost": 70, "trlic": 118}


# ---------------------------------------------------------------------------
# Gate library: each block as Toffoli-type gates, and its defining function
# ---------------------------------------------------------------------------
def feynman(a: int, b: int) -> List[Gate]:
    """Feynman gate: (A, B) -> (A, A xor B)."""
    return [CNOT(a, b)]


def peres(a: int, b: int, c: int) -> List[Gate]:
    """Peres gate: (A, B, C) -> (A, A xor B, AB xor C)."""
    return [TOF(a, b, c), CNOT(a, b)]


def tsg(a: int, b: int, c: int, d: int) -> List[Gate]:
    """TSG gate of Thapliyal and Srinivas: (A, B, C, D) -> (A, Q, Q xor D, QD xor AB xor C)
    with Q = A'C' xor B'. The outputs appear on lines a, b, d, c in that order."""
    return [TOF(a, b, c), CNOT(a, b), G([c], [a], b), TOF(b, d, c), CNOT(b, d)]


def r2(a: int, b: int, c: int, d: int) -> List[Gate]:
    """R2 gate: (A, B, C, D) -> (A, B, C, D xor A xor B xor C)."""
    return [CNOT(a, d), CNOT(b, d), CNOT(c, d)]


def feynman_ref(a, b):
    return a, a ^ b


def peres_ref(a, b, c):
    return a, a ^ b, (a & b) ^ c


def tsg_ref(a, b, c, d):
    q = ((1 - a) & (1 - c)) ^ (1 - b)
    return a, q, q ^ d, (q & d) ^ (a & b) ^ c


def r2_ref(a, b, c, d):
    return a, b, c, d ^ a ^ b ^ c


# ---------------------------------------------------------------------------
# A design: circuit plus the meaning of its lines
# ---------------------------------------------------------------------------
@dataclass
class Design:
    name: str
    circuit: Circuit
    line_names: List[str]
    inputs: Dict[str, int]                 # primary input name -> line
    ancilla: List[int]                     # lines that must start at 0
    outputs: Dict[str, int]                # primary output name -> line
    blocks: List[Tuple[str, Tuple[int, ...]]] = field(default_factory=list)

    @property
    def n(self) -> int:
        return self.circuit.n

    @property
    def garbage(self) -> List[int]:
        """Every output line that is not a primary output."""
        used = set(self.outputs.values())
        return [l for l in range(self.n) if l not in used]

    def block_count(self) -> int:
        return len(self.blocks)

    def block_cost(self) -> int:
        """Quantum cost with the block costs stated in the adder paper."""
        return sum(PAPER_BLOCK_COST[kind] for kind, _ in self.blocks)

    def block_summary(self) -> str:
        kinds = [kind for kind, _ in self.blocks]
        return ", ".join(f"{kinds.count(k)} {k}" for k in ("Peres", "Feynman", "TSG", "R2")
                         if k in kinds)

    def with_circuit(self, circuit: Circuit, name: str) -> "Design":
        """The same interface around another circuit (e.g. an optimized one)."""
        return Design(name, circuit, self.line_names, self.inputs, self.ancilla,
                      self.outputs, [])

    def run(self, values: Dict[str, int]) -> Dict[str, int]:
        """Simulate with the given primary input bits; ancilla lines start at 0."""
        state = sum(values[name] << line for name, line in self.inputs.items())
        out = self.circuit.apply(state)
        return {name: (out >> line) & 1 for name, line in self.outputs.items()}


class _Builder:
    def __init__(self, names: Sequence[str]) -> None:
        self.names = list(names)
        self.index = {name: i for i, name in enumerate(self.names)}
        self.gates: List[Gate] = []
        self.blocks: List[Tuple[str, Tuple[int, ...]]] = []

    def add(self, kind: str, *line_names: str) -> None:
        lines = tuple(self.index[name] for name in line_names)
        maker = {"Feynman": feynman, "Peres": peres, "TSG": tsg, "R2": r2}[kind]
        self.gates += maker(*lines)
        self.blocks.append((kind, lines))


# ---------------------------------------------------------------------------
# Carry look-ahead adder (reconstruction of Kumar et al.)
# ---------------------------------------------------------------------------
def cla_adder(carry_out: bool = False) -> Design:
    """4-bit reversible carry look-ahead adder.

    carry_out=False  the paper's block inventory: sum bits S0..S3 only.
    carry_out=True   the last Feynman gate is replaced by a Peres gate that
                     also produces the final carry C4 = G3 xor P3 C3.
    """
    names = ([f"a{i}" for i in range(4)] + [f"b{i}" for i in range(4)] + ["c0"]
             + [f"g{i}" for i in range(4)]            # G_i
             + ["k0", "k1", "k2", "k3"]               # copies
             + [f"t{i}" for i in range(1, 7)]         # product terms
             + ["z"])                                 # constant input of the TSG gate
    b = _Builder(names)
    for i in range(4):                                # PGA, Peres part: b_i = P_i, g_i = G_i
        b.add("Peres", f"a{i}", f"b{i}", f"g{i}")
    b.add("Feynman", "c0", "k0")                      # copy C0
    b.add("Feynman", "b0", "c0")                      # PGA 0, Feynman part: c0 = S0
    b.add("Peres", "b0", "k0", "t1")                  # t1 = P0 C0
    b.add("Feynman", "g0", "k1")                      # copy G0
    b.add("Feynman", "t1", "k1")                      # k1 = C1 = G0 xor P0 C0
    b.add("Peres", "b1", "g0", "t2")                  # t2 = P1 G0
    b.add("Peres", "b1", "t1", "t3")                  # t3 = P1 P0 C0
    b.add("Feynman", "b1", "k1")                      # PGA 1, Feynman part: k1 = S1
    b.add("Feynman", "g1", "k2")                      # copy G1
    b.add("Feynman", "t2", "k3")                      # copy P1 G0
    b.add("TSG", "t3", "k3", "z", "k2")               # k2 = C2 (three-input XOR)
    b.add("Peres", "b2", "g1", "t4")                  # t4 = P2 G1
    b.add("Peres", "b2", "t2", "t5")                  # t5 = P2 P1 G0
    b.add("Peres", "b2", "t3", "t6")                  # t6 = P2 P1 P0 C0
    b.add("Feynman", "b2", "k2")                      # PGA 2, Feynman part: k2 = S2
    b.add("R2", "t4", "t5", "t6", "g2")               # g2 = C3 (four-input XOR)
    outputs = {"s0": "c0", "s1": "k1", "s2": "k2", "s3": "g2"}
    if carry_out:
        b.add("Peres", "b3", "g2", "g3")              # g2 = S3, g3 = C4
        outputs["c4"] = "g3"
    else:
        b.add("Feynman", "b3", "g2")                  # PGA 3, Feynman part: g2 = S3
    ix = b.index
    return Design(
        name="CLA with carry-out" if carry_out else "CLA (paper style)",
        circuit=Circuit(len(names), b.gates), line_names=names,
        inputs={n: ix[n] for n in names[:9]},
        ancilla=[ix[n] for n in names[9:]],
        outputs={k: ix[v] for k, v in outputs.items()}, blocks=b.blocks)


# ---------------------------------------------------------------------------
# Ripple-carry adder: two Peres gates per bit
# ---------------------------------------------------------------------------
def rca_adder() -> Design:
    """4-bit reversible ripple-carry adder with carry-out.

    Per bit: Peres(A_i, B_i, 0) gives P_i and G_i, then Peres(P_i, C_i, G_i)
    gives S_i and C_(i+1) = G_i xor P_i C_i.
    """
    names = ([f"a{i}" for i in range(4)] + [f"b{i}" for i in range(4)] + ["c0"]
             + [f"z{i}" for i in range(4)])
    b = _Builder(names)
    carry = "c0"
    outputs = {}
    for i in range(4):
        b.add("Peres", f"a{i}", f"b{i}", f"z{i}")
        b.add("Peres", f"b{i}", carry, f"z{i}")
        outputs[f"s{i}"] = carry
        carry = f"z{i}"
    outputs["c4"] = carry
    ix = b.index
    return Design(
        name="Ripple-carry (Peres)", circuit=Circuit(len(names), b.gates), line_names=names,
        inputs={n: ix[n] for n in names[:9]}, ancilla=[ix[n] for n in names[9:]],
        outputs={k: ix[v] for k, v in outputs.items()}, blocks=b.blocks)


# ---------------------------------------------------------------------------
# Checks and measures
# ---------------------------------------------------------------------------
def check_adder(design: Design) -> Tuple[int, int]:
    """Test all 512 input combinations. Returns (cases tested, cases wrong)."""
    wrong = 0
    for a in range(16):
        for b in range(16):
            for c0 in range(2):
                values = {f"a{i}": (a >> i) & 1 for i in range(4)}
                values.update({f"b{i}": (b >> i) & 1 for i in range(4)})
                values["c0"] = c0
                out = design.run(values)
                total = a + b + c0
                good = all(out[f"s{i}"] == (total >> i) & 1 for i in range(4))
                if "c4" in out:
                    good = good and out["c4"] == (total >> 4) & 1
                wrong += not good
    return 512, wrong


def add(design: Design, a: int, b: int, c0: int = 0):
    """Add two 4-bit numbers with a design. Returns (sum, carry); carry is None
    when the design has no carry output."""
    values = {f"a{i}": (a >> i) & 1 for i in range(4)}
    values.update({f"b{i}": (b >> i) & 1 for i in range(4)})
    values["c0"] = c0
    out = design.run(values)
    total = sum(out[f"s{i}"] << i for i in range(4))
    return total, out.get("c4")


def depth(circuit: Circuit) -> int:
    """Number of time steps when gates on different lines run in parallel."""
    level = [0] * circuit.n
    for g in circuit:
        step = 1 + max(level[l] for l in g.lines)
        for l in g.lines:
            level[l] = step
    return max(level, default=0)

## Step 3: Fast simulation of all inputs

In [ ]:
%%writefile bitsim.py
"""Bit-parallel simulation of reversible circuits (Phase 4).

revcircuit.py simulates one input at a time, which is too slow for a
24-line adder (16.7 million inputs). Here every line holds one large
integer whose bit s is the value of that line for input state s, so a gate
is applied to all 2^n inputs at once with a few integer operations.

This makes an exhaustive equivalence check practical up to about 26 lines.
"""
from __future__ import annotations

from typing import List

from revcircuit import Circuit


def input_vectors(n: int) -> List[int]:
    """vectors[i] has bit s equal to bit i of s, for s = 0 .. 2^n - 1."""
    size = 1 << n
    vectors = []
    for i in range(n):
        half = 1 << i
        v = ((1 << half) - 1) << half            # 'half' zeros, then 'half' ones
        length = 2 * half
        while length < size:                     # repeat the pattern to full length
            v |= v << length
            length *= 2
        vectors.append(v)
    return vectors


def output_vectors(circuit: Circuit) -> List[int]:
    """Truth vector of every output line over all 2^n inputs."""
    n = circuit.n
    mask = (1 << (1 << n)) - 1
    lines = input_vectors(n)
    for g in circuit:
        active = mask
        for l in g.pos:
            active &= lines[l]
        for l in g.neg:
            active &= ~lines[l]
        lines[g.target] ^= active & mask
    return lines


def equivalent_all_inputs(a: Circuit, b: Circuit) -> bool:
    """True when the two circuits agree on every line for every one of the 2^n inputs."""
    return a.n == b.n and output_vectors(a) == output_vectors(b)

## Step 4: Verification script

In [ ]:
%%writefile verify_phase4.py
"""Phase 4 verification: reversible adders and the rule-based optimizer.

  A. Gate library: each block equals its defining function.
  B. The adders add correctly for all 512 input combinations.
  C. Resources of each design, beside the figures the adder paper reports.
  D. The optimizer on the adders, checked on every input of every line.
  E. Search for hidden redundancy: reorder each adder at random and optimize again.
  F. Control experiment: redundancy planted in the adder is removed again.

Run:  python verify_phase4.py
"""
from __future__ import annotations

import random
import sys
import time
from itertools import product

from revcircuit import CNOT, Circuit, G, TOF, X
from optimizer import circuit_cost, commute_tag, optimize, pass_gate
from adders import (PAPER_BLOCK_COST, PAPER_REPORTED, check_adder, cla_adder,
                    depth, feynman, feynman_ref, peres, peres_ref, r2, r2_ref,
                    rca_adder, tsg, tsg_ref)
from bitsim import equivalent_all_inputs

# The circuit drawings use Unicode symbols; make sure they print on Windows too.
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

rng = random.Random(2026)
failures = 0


def report(label: str, ok: bool, detail: str = "") -> None:
    global failures
    failures += not ok
    print(f"  {label:<50}{'PASS' if ok else 'FAIL'}" + (f"  {detail}" if detail else ""))


# ---------------------------------------------------------------------------
# A. Gate library
# ---------------------------------------------------------------------------
def part_a() -> None:
    print("A. Gate library (each block as Toffoli-type gates)")
    print(f"  {'Block':<9}{'Lines':>6}{'Gates':>7}{'Cost, this project':>20}"
          f"{'Cost, adder paper':>19}  Result")
    library = (("Feynman", feynman, feynman_ref, 2, None),
               ("Peres", peres, peres_ref, 3, None),
               ("TSG", tsg, tsg_ref, 4, (0, 1, 3, 2)),      # outputs on lines a, b, d, c
               ("R2", r2, r2_ref, 4, None))
    for name, maker, ref, k, order in library:
        circuit = Circuit(k, maker(*range(k)))
        ok, images = True, set()
        for bits in product((0, 1), repeat=k):
            state = sum(bit << i for i, bit in enumerate(bits))
            out = circuit.apply(state)
            got = tuple((out >> i) & 1 for i in range(k))
            want = ref(*bits)
            if order:
                want = tuple(want[j] for j in order)
            ok &= got == want
            images.add(got)
        ok &= len(images) == 1 << k                           # the block is reversible
        global failures
        failures += not ok
        print(f"  {name:<9}{k:>6}{len(circuit):>7}{circuit_cost(circuit):>20}"
              f"{PAPER_BLOCK_COST[name]:>19}  {'PASS' if ok else 'FAIL'}")
    print("  The two cost columns differ because the adder paper prices each block as a")
    print("  unit, while this project prices the Toffoli-type gates inside it.\n")


# ---------------------------------------------------------------------------
# B and C. The adders
# ---------------------------------------------------------------------------
def part_b(designs) -> None:
    print("B. Adders: all 512 input combinations (ancilla lines start at 0)")
    for d in designs:
        cases, wrong = check_adder(d)
        outs = ", ".join(sorted(d.outputs))
        report(f"{d.name}: outputs {outs}", wrong == 0, f"{cases - wrong}/{cases} correct")
    print()


def part_c(designs) -> None:
    print("C. Resources")
    print(f"  {'Design':<24}{'Lines':>6}{'Blocks':>8}{'Ancilla':>9}{'Garbage':>9}"
          f"{'Cost (paper':>13}{'Gates':>7}{'Cost (this':>12}{'Depth':>7}")
    print(f"  {'':<24}{'':>6}{'':>8}{'':>9}{'':>9}{'prices)':>13}{'':>7}{'project)':>12}{'':>7}")
    p = PAPER_REPORTED
    print(f"  {'Reported in adder paper':<24}{'-':>6}{p['gates']:>8}{p['ancilla']:>9}"
          f"{p['garbage']:>9}{p['cost']:>13}{'-':>7}{'-':>12}{'-':>7}")
    for d in designs:
        print(f"  {d.name:<24}{d.n:>6}{d.block_count():>8}{len(d.ancilla):>9}"
              f"{len(d.garbage):>9}{d.block_cost():>13}{len(d.circuit):>7}"
              f"{circuit_cost(d.circuit):>12}{depth(d.circuit):>7}")
    print("  Blocks   = Feynman, Peres, TSG and R2 gates, each counted once.")
    print("  Garbage  = every output line that is not a sum or carry output.")
    print("  Gates    = Toffoli-type gates after decomposing the blocks.")
    print("  Depth    = time steps of this netlist when gates on separate lines run together.")
    for d in designs:
        print(f"  {d.name}: {d.block_summary()}")
    print()


# ---------------------------------------------------------------------------
# D. Optimizer on the adders
# ---------------------------------------------------------------------------
def part_d(designs) -> None:
    print("D. Rule-based optimizer on the adders (every line treated as a real input and output)")
    print(f"  {'Design':<24}{'Objective':<11}{'Gates':>12}{'Cost':>12}{'Depth':>10}"
          f"{'Rewrites':>10}  Result")
    for d in designs:
        for objective in ("cost", "gates"):
            result = optimize(d.circuit, objective=objective)
            same = equivalent_all_inputs(d.circuit, result.circuit)
            adds = check_adder(d.with_circuit(result.circuit, d.name))[1] == 0
            ok = same and adds and result.cost_after <= result.cost_before
            global failures
            failures += not ok
            print(f"  {d.name:<24}{objective:<11}"
                  f"{f'{result.gates_before} -> {result.gates_after}':>12}"
                  f"{f'{result.cost_before} -> {result.cost_after}':>12}"
                  f"{f'{depth(d.circuit)} -> {depth(result.circuit)}':>10}"
                  f"{result.rewrites:>10}  {'PASS' if ok else 'FAIL'}")
    print("  PASS = identical to the original on all 2^n inputs of all lines, and still adds.\n")


# ---------------------------------------------------------------------------
# E. Hidden redundancy
# ---------------------------------------------------------------------------
def reorder(circuit: Circuit, steps: int) -> Circuit:
    """Random function-preserving moves that keep the gate count: commute, or Rule 9."""
    gates = list(circuit.gates)
    for _ in range(steps):
        i = rng.randrange(len(gates) - 1)
        a, b = gates[i], gates[i + 1]
        if commute_tag(a, b) is not None:
            gates[i], gates[i + 1] = b, a
        else:
            passed = pass_gate(a, b)
            if passed is not None and passed[3] == 0:        # Rule 9: no extra gate
                gates[i:i + 2] = passed[0] + [passed[1]]
    return Circuit(circuit.n, gates)


def part_e(designs) -> None:
    print("E. Search for hidden redundancy (random reordering, then optimize again)")
    trials = 200
    for d in designs:
        base = circuit_cost(d.circuit)
        best, ok = base, True
        for k in range(trials):
            shuffled = reorder(d.circuit, 400)
            result = optimize(shuffled, extra_gates=2)
            best = min(best, result.cost_after)
            if k % 50 == 0:                                   # spot-check full equivalence
                ok &= equivalent_all_inputs(d.circuit, result.circuit)
        report(f"{d.name}: {trials} random orderings", ok,
               f"lowest cost found {best} (original {base})")
    print()


# ---------------------------------------------------------------------------
# F. Planted redundancy
# ---------------------------------------------------------------------------
def plant(circuit: Circuit, pairs: int) -> Circuit:
    """Insert `pairs` cancelling gate pairs (Rule 1 backwards), then drift them apart."""
    gates = list(circuit.gates)
    n = circuit.n
    for _ in range(pairs):
        kind = rng.choice((0, 1, 2))
        lines = rng.sample(range(n), kind + 1)
        g = G(lines[1:], [], lines[0])
        i = rng.randrange(len(gates) + 1)
        gates[i:i] = [g, g]
    return reorder(Circuit(n, gates), 600)


def part_f(designs) -> None:
    print("F. Control experiment: 15 cancelling gate pairs planted in each adder")
    print(f"  {'Design':<24}{'Trials':>7}{'Avg cost planted':>18}{'Avg cost after':>16}"
          f"{'Original':>10}{'Fully recovered':>17}  Result")
    trials = 30
    for d in designs:
        base = circuit_cost(d.circuit)
        planted_sum = after_sum = recovered = 0
        ok = True
        for k in range(trials):
            padded = plant(d.circuit, 15)
            result = optimize(padded)
            planted_sum += circuit_cost(padded)
            after_sum += result.cost_after
            recovered += result.cost_after <= base
            ok &= check_adder(d.with_circuit(result.circuit, d.name))[1] == 0
            if k % 10 == 0:
                ok &= equivalent_all_inputs(d.circuit, result.circuit)
        global failures
        failures += not ok
        print(f"  {d.name:<24}{trials:>7}{planted_sum / trials:>18.1f}{after_sum / trials:>16.1f}"
              f"{base:>10}{100 * recovered / trials:>16.1f}%  {'PASS' if ok else 'FAIL'}")
    print("  PASS = every optimized circuit still adds correctly.\n")


def main() -> int:
    start = time.time()
    print("Phase 4 verification: reversible adders\n")
    designs = [cla_adder(), cla_adder(carry_out=True), rca_adder()]
    part_a()
    part_b(designs)
    part_c(designs)
    part_d(designs)
    part_e(designs)
    part_f(designs)
    print(f"Time: {time.time() - start:.1f} s")
    print("\nAll checks passed." if failures == 0 else f"\n{failures} check(s) FAILED.")
    return 1 if failures else 0


if __name__ == "__main__":
    sys.exit(main())

## Step 5: Run the verification

Every check should show **PASS**, and the output should end with `All checks passed.`

In [ ]:
!python verify_phase4.py

## Step 6: Try it yourself

Change the numbers in `examples` and run the cell again. Each number must be between 0 and 15.

Drawing symbols: `⊕` target, `●` positive control, `○` negative control.

In [ ]:
import importlib
import revcircuit, rules, optimizer, adders, bitsim
for m in (revcircuit, rules, optimizer, adders, bitsim):
    importlib.reload(m)

from adders import cla_adder, rca_adder, add, depth
from optimizer import optimize, circuit_cost

cla = cla_adder(carry_out=True)
rca = rca_adder()

examples = [(5, 3, 0), (15, 1, 0), (10, 5, 0), (9, 9, 1)]     # (A, B, carry-in)
for a, b, c0 in examples:
    s1, c1 = add(cla, a, b, c0)
    s2, c2 = add(rca, a, b, c0)
    print(f"{a:>2} + {b:>2} + {c0}:  carry look-ahead sum {s1:>2} carry {c1}   "
          f"ripple-carry sum {s2:>2} carry {c2}")

print()
print("Blocks of the carry look-ahead adder, in order:")
for kind, lines in cla.blocks:
    print(f"  {kind:<8}", ", ".join(cla.line_names[l] for l in lines))

print()
print("Ripple-carry adder as Toffoli-type gates "
      f"({len(rca.circuit)} gates, cost {circuit_cost(rca.circuit)}, depth {depth(rca.circuit)}):")
for name, row in zip(rca.line_names, rca.circuit.draw().splitlines()):
    print(f"{name:>3} {row.split(':', 1)[1].lstrip()}")